In [2]:
print("Spark is working")

StatementMeta(, 16ebc071-7d48-4bc2-ad66-1648fb8441ba, 4, Finished, Available, Finished, False)

Spark is working


In [3]:
display(spark.sql("SHOW DATABASES"))

StatementMeta(, 16ebc071-7d48-4bc2-ad66-1648fb8441ba, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e449918e-e8bd-49b1-9f22-f90e07b61cc4)

In [4]:
df = spark.read.option("header", "true").csv("Files/test_results.csv")

display(df)

StatementMeta(, 16ebc071-7d48-4bc2-ad66-1648fb8441ba, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cc6f0421-1b45-4a1a-a072-10593508d704)

In [5]:
from pyspark.sql.functions import col

total_tests = df.count()

passed_tests = df.filter(col("Status") == "PASS").count()

failed_tests = df.filter(col("Status") == "FAIL").count()

print(f"Total Tests: {total_tests}")
print(f"Passed Tests: {passed_tests}")
print(f"Failed Tests: {failed_tests}")

StatementMeta(, 16ebc071-7d48-4bc2-ad66-1648fb8441ba, 7, Finished, Available, Finished, False)

Total Tests: 10
Passed Tests: 7
Failed Tests: 3


In [6]:
pass_percentage = (passed_tests / total_tests) * 100

print(f"Pass Percentage: {pass_percentage:.2f}%")

StatementMeta(, 16ebc071-7d48-4bc2-ad66-1648fb8441ba, 8, Finished, Available, Finished, False)

Pass Percentage: 70.00%


In [8]:
summary_data = [
    ("Total Tests", str(total_tests)),
    ("Passed Tests", str(passed_tests)),
    ("Failed Tests", str(failed_tests)),
    ("Pass Percentage", f"{pass_percentage:.2f}%")
]

summary_df = spark.createDataFrame(summary_data, ["Metric", "Value"])

display(summary_df)


StatementMeta(, 16ebc071-7d48-4bc2-ad66-1648fb8441ba, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 969efd7c-7f25-46a2-a78a-f5b7809f536b)

In [9]:
status_summary = (
    df.groupBy("Status")
      .count()
)

display(status_summary)

StatementMeta(, 16ebc071-7d48-4bc2-ad66-1648fb8441ba, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1974c0e2-b3ce-4d37-92f9-468f4e5a7eb7)

In [10]:
status_summary.write.mode("overwrite").saveAsTable("test_status_summary")

StatementMeta(, 16ebc071-7d48-4bc2-ad66-1648fb8441ba, 12, Finished, Available, Finished, False)

In [11]:
display(spark.sql("SHOW TABLES"))

StatementMeta(, 16ebc071-7d48-4bc2-ad66-1648fb8441ba, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2196072b-b605-4890-8137-74845dcb4a4c)

In [12]:
%%sql
SELECT *
FROM test_status_summary

StatementMeta(, 16ebc071-7d48-4bc2-ad66-1648fb8441ba, 14, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

In [3]:
from pyspark.sql import Row
import random

tests = [
    "LoginTest",
    "LogoutTest",
    "SearchTest",
    "PaymentTest",
    "ProfileTest",
    "RegistrationTest",
    "CartTest",
    "CheckoutTest",
    "NotificationTest",
    "SettingsTest"
]

rows = []

for build in range(101, 106):
    for test in tests:
        rows.append(
            Row(
                TestName=test,
                Status=random.choice(["PASS", "PASS", "PASS", "FAIL"]),
                Duration=random.randint(2, 25),
                BuildNumber=f"Build_{build}"
            )
        )

big_df = spark.createDataFrame(rows)

display(big_df)

big_df.write.mode("overwrite").saveAsTable("test_executions")

StatementMeta(, 3b02f912-b704-4f1f-b377-daf2e3ee442c, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6b797619-006b-498f-a925-1bd358931cc7)

In [4]:
display(spark.sql("SHOW TABLES"))

StatementMeta(, 3b02f912-b704-4f1f-b377-daf2e3ee442c, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2bf5aef2-9152-4628-9cdd-7b1704bde7b2)

In [5]:
big_df = spark.table("test_executions")

display(big_df)

StatementMeta(, 3b02f912-b704-4f1f-b377-daf2e3ee442c, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7459015d-6638-42a7-8591-20f8250b3422)

In [6]:
from pyspark.sql.functions import count, sum, when, round

build_summary = (
    big_df
    .groupBy("BuildNumber")
    .agg(
        count("*").alias("TotalTests"),
        sum(
            when(big_df.Status == "PASS", 1).otherwise(0)
        ).alias("PassedTests")
    )
)

build_summary = build_summary.withColumn(
    "PassPercentage",
    round(
        (build_summary.PassedTests / build_summary.TotalTests) * 100,
        2
    )
)

display(build_summary)

StatementMeta(, 3b02f912-b704-4f1f-b377-daf2e3ee442c, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 47406d33-692e-442c-8891-95131dd12e13)

In [7]:
from pyspark.sql.functions import count

failing_tests = (
    big_df
    .filter(big_df.Status == "FAIL")
    .groupBy("TestName")
    .agg(count("*").alias("FailureCount"))
    .orderBy("FailureCount", ascending=False)
)

display(failing_tests)

StatementMeta(, 3b02f912-b704-4f1f-b377-daf2e3ee442c, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 48a50486-6a45-4290-b67b-8c6008f4bd66)

In [8]:
from pyspark.sql.functions import avg, round

slow_tests = (
    big_df
    .groupBy("TestName")
    .agg(
        round(avg("Duration"), 2).alias("AvgDuration")
    )
    .orderBy("AvgDuration", ascending=False)
)

display(slow_tests)

StatementMeta(, 3b02f912-b704-4f1f-b377-daf2e3ee442c, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 80106e51-13ab-4b5e-bc44-49da614a5d2f)